In [1]:
from google.colab import files
import zipfile
import os

# Upload the zip file
print("Please upload the zip file:")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'User uploaded file "{fn}" ({len(uploaded[fn])} bytes)')
    # Unzip the file
    with zipfile.ZipFile(fn, 'r') as zip_ref:
        zip_ref.extractall(".")
    print(f'Successfully unzipped {fn}')
    # Remove the uploaded zip file after extraction to keep the directory clean
    os.remove(fn)

Please upload the zip file:


Saving archive.zip to archive.zip
User uploaded file "archive.zip" (11949309 bytes)
Successfully unzipped archive.zip


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── Standard library ──────────────────────────────────────────────────────────
from __future__ import annotations
import csv
import hashlib
import html
import json
import os
import pickle
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")          # keep Colab output clean

# ── Third-party (always available) ───────────────────────────────────────────
import numpy as np
import torch
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from cluster_store import save_cluster_data

In [4]:
# ── GPU / CPU conditional imports ─────────────────────────────────────────
CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:
    try:
        import cupy as cp
        from cuml.decomposition import PCA as _PCA
        from cuml.manifold import UMAP as _UMAP
        from cuml.cluster import HDBSCAN as _HDBSCAN
        USE_GPU = True
        print("✓ GPU detected")
    except ImportError:
        USE_GPU = False
else:
    USE_GPU = False

if not USE_GPU:
    from sklearn.decomposition import PCA as _PCA
    from umap import UMAP as _UMAP
    from hdbscan import HDBSCAN as _HDBSCAN

# ── Constants ─────────────────────────────────────────────────────────────────
CSV_FILE = Path("/content/train.csv")

PCA_DIMS          = 150
CLUSTER_UMAP_DIMS = 50

_DRIVE_CACHE = "/content/drive/MyDrive/clustering_cache"
_LOCAL_CACHE = "/tmp/clustering_cache"

✓ RAPIDS / cuML detected — using GPU pipeline


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────

# Compiled once at module load — cheap to reuse
_RE_SOURCE_SUFFIX = re.compile(r'\s+-\s+\S+\.\S+$')   # " - www.site.com" at end of title
_RE_HTML_TAG      = re.compile(r'<[^>]+>')
_RE_NONALPHA      = re.compile(r'[^a-zA-Z0-9\s\'\-]')
_RE_SPACES        = re.compile(r'\s+')
# Strip bare short-digit tokens (1–3 digits) that are space-bounded.
# These come from HTML entity residue (&#39; → ' → 39 after apostrophe removal)
# or formatting fragments. Hyphen-attached digits like COVID-19 and G7 are kept.
# 4-digit numbers (years, etc.) are never touched.
_RE_SHORT_NUM     = re.compile(r'(?<![a-zA-Z0-9\-])\d{1,3}(?![a-zA-Z0-9\-])')
# Collapse thousands-separator commas BEFORE punctuation stripping:
# "1,000" → "1000", "50,000" → "50000". Applied twice for millions.
_RE_THOUSANDS     = re.compile(r'(\d),(\d)')


def clean_title(raw: str) -> str:
    """
    Normalise a news headline for both TF-IDF and embedding.

    Steps (in order):
      1. html.unescape x2 : handles double-encoded entities (common in RSS)
         &#39; → '   &amp; → &   &lt; → <   &gt; → >
         &amp;#39; → &#39; → ' (double-encoded)
      2. Strip source suffix: "Some Title - www.washingtonpost.com" → "Some Title"
      3. Strip residual HTML tags (<b>, <i>, etc.)
      4. Collapse formatted numbers: "1 000 jobs" → "1000 jobs"
         Prevents "000" token surviving after comma-stripping.
      5. Remove punctuation that isn't apostrophe or hyphen
         (keeps contractions and hyphenated words intact)
      6. Remove bare short-digit tokens (1–3 digits) left by HTML entity
         residue, e.g. the "39" in "don 39t" that comes from &#39; → ' → 39
         after apostrophe stripping. Real years (4 digits) are kept.
      7. Collapse whitespace and strip
    """
    s = html.unescape(raw)                  # first pass: &amp;#39; → &#39;
    s = html.unescape(s)                    # second pass: &#39; → '
    s = _RE_SOURCE_SUFFIX.sub('', s)        # "Title - site.com" → "Title"
    s = _RE_HTML_TAG.sub(' ', s)            # <b>foo</b> → foo
    s = _RE_THOUSANDS.sub(r'\1\2', s)      # "1,000" → "1000" before comma-strip
    s = _RE_THOUSANDS.sub(r'\1\2', s)      # second pass for "1,000,000"
    s = _RE_NONALPHA.sub(' ', s)            # remove remaining punctuation / symbols
    s = _RE_SHORT_NUM.sub(' ', s)           # remove bare 1-3 digit tokens (entity residue)
    s = _RE_SPACES.sub(' ', s)              # collapse whitespace
    return s.strip()


def load_titles_from_csv(csv_file: Path = CSV_FILE) -> list[str]:
    """Load and clean non-empty values from the 'Title' column of a CSV file."""
    titles: list[str] = []
    with csv_file.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw = (row.get("Title") or "").strip()
            if not raw:
                continue
            cleaned = clean_title(raw)
            if cleaned:                 # skip titles that become empty after cleaning
                titles.append(cleaned)
    return titles


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cache helpers
# ─────────────────────────────────────────────────────────────────────────────

def _resolve_cache_dir() -> str:
    """
    Return the cache directory to use.
    Prefer Google Drive (persistent across sessions) when it is mounted.
    Fall back to /tmp (lost on session restart, but no error).
    """
    if os.path.isdir("/content/drive/MyDrive"):
        cache_dir = _DRIVE_CACHE
    else:
        cache_dir = _LOCAL_CACHE
    os.makedirs(cache_dir, exist_ok=True)
    return cache_dir


CACHE_DIR = _resolve_cache_dir()


def _cache_path(name: str) -> str:
    return os.path.join(CACHE_DIR, f"{name}.pkl")


def save_cache(name: str, obj) -> None:
    path = _cache_path(name)
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_cache(name: str):
    path = _cache_path(name)
    if os.path.exists(path):
        with open(path, "rb") as f:
            obj = pickle.load(f)
        return obj
    return None


# ─────────────────────────────────────────────────────────────────────────────
# Quality metrics
# ─────────────────────────────────────────────────────────────────────────────

def _compute_silhouette(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """Compute silhouette score (range -1 to 1, higher is better)."""
    from sklearn.metrics import silhouette_score
    # Remove noise points (label == -1)
    mask = labels != -1
    if mask.sum() < 2:
        return 0.0
    try:
        return silhouette_score(embeddings[mask], labels[mask], sample_size=min(5000, mask.sum()))
    except Exception:
        return 0.0


def _cluster_size_stats(labels: np.ndarray) -> dict:
    """Compute cluster size statistics, excluding noise."""
    from collections import Counter
    counts = Counter(labels[labels != -1])
    if not counts:
        return {"min": 0, "max": 0, "median": 0, "mean": 0.0, "std": 0.0}
    sizes = np.array(list(counts.values()))
    return {
        "min": int(sizes.min()),
        "max": int(sizes.max()),
        "median": int(np.median(sizes)),
        "mean": float(sizes.mean()),
        "std": float(sizes.std()),
    }


def _noise_percentage(labels: np.ndarray) -> float:
    """Percentage of points labeled as noise (-1)."""
    return 100.0 * (labels == -1).sum() / len(labels)


def _group_quality_summary(embeddings: np.ndarray, group_labels: np.ndarray, sub_labels: np.ndarray, group_id: int) -> str:
    """Compute quality for a single top-level group."""
    mask = group_labels == group_id
    if not mask.any():
        return ""
    
    g_labels = sub_labels[mask]
    n_subs = len(set(g_labels) - {-1})
    noise_pct = _noise_percentage(g_labels)
    
    try:
        sil = _compute_silhouette(embeddings[mask], g_labels)
        return f"({n_subs} sub, sil={sil:.2f}, noise={noise_pct:.1f}%)"
    except Exception:
        return f"({n_subs} sub, noise={noise_pct:.1f}%)"

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────────────────────────────────────

def main() -> None:
    # ── 0. Load data ──────────────────────────────────────────────────────────
    sentences = load_titles_from_csv()
    if len(sentences) < 2:
        raise ValueError("Need at least 2 titles to cluster.")

    N  = len(sentences)
    fp = data_fingerprint(sentences)
    print(f"\n{'═'*50}")
    print(f"CLUSTERING PIPELINE")
    print(f"{'═'*50}")
    print(f"Dataset: {N:,} titles")
    print(f"Cache:   {CACHE_DIR}\n")

    # ── 1. Embeddings ─────────────────────────────────────────────────────────
    emb_key    = f"embeddings_{fp}"
    embeddings = load_cache(emb_key)

    if embeddings is None:
        device = "cuda" if CUDA_AVAILABLE else "cpu"
        print(f"[1] Computing embeddings ({device}) …")
        model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
        embeddings = model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=64 if CUDA_AVAILABLE else 16,
            show_progress_bar=True,
        )
        del model
        if CUDA_AVAILABLE:
            torch.cuda.empty_cache()
        save_cache(emb_key, embeddings)
    else:
        print(f"[1] Embeddings loaded from cache")

    # ── 2. Top-level clustering ───────────────────────────────────────────────
    print(f"[2] Top-level clustering …")
    raw_group_labels = semantic_subclusters(embeddings, group_size=N)

    group_ids = sorted(set(raw_group_labels) - {-1})
    if not group_ids:
        group_labels = np.zeros(N, dtype=int)
        group_topics = ["Cluster 0"]
    else:
        group_labels = np.full(N, -1, dtype=int)
        group_topics = []
        for new_id, old_id in enumerate(group_ids):
            mask = raw_group_labels == old_id
            group_labels[mask] = new_id
            group_topics.append(infer_topic([sentences[i] for i in np.where(mask)[0]]))
    
    # Quality metrics
    sil_top = _compute_silhouette(embeddings, group_labels)
    sizes_top = _cluster_size_stats(group_labels)
    print(f"    → {len(group_topics)} clusters")
    print(f"    └ Quality: silhouette={sil_top:.2f}, sizes: {sizes_top['min']}–{sizes_top['max']} (median {sizes_top['median']})")

    # ── 3. Level 2 — semantic sub-clusters inside each top-level cluster ──────
    sub_key    = f"sub_labels_v2_{fp}"
    sub_labels = load_cache(sub_key)

    print(f"[3] Sub-clustering within clusters …")
    if sub_labels is None:
        sub_labels = np.full(N, -1, dtype=int)

        for g in range(len(group_topics)):
            idx = np.where(group_labels == g)[0]
            if len(idx) == 0:
                continue

            subs = semantic_subclusters(embeddings[idx], group_size=len(idx))
            sub_labels[idx] = subs

            quality = _group_quality_summary(embeddings, group_labels, sub_labels, g)
            print(f"    Group {g}: {quality}")

        save_cache(sub_key, sub_labels)

    # ── 4. Combined labels & topics ───────────────────────────────────────────
    combined_key    = f"combined_v2_{fp}"
    cached_combined = load_cache(combined_key)

    if cached_combined is None:
        combined_labels = np.where(
            (group_labels == -1) | (sub_labels == -1),
            -1,
            group_labels * 10_000 + sub_labels,
        ).astype(int)

        combined_topics: dict[int, str] = {}
        for clabel in sorted(set(combined_labels)):
            if clabel == -1:
                continue
            mask  = combined_labels == clabel
            topic = infer_topic([sentences[i] for i in np.where(mask)[0]])
            g     = clabel // 10_000
            combined_topics[clabel] = f"[{group_topics[g][:25]}] › {topic}"

        save_cache(combined_key, (combined_labels, combined_topics))
    else:
        combined_labels, combined_topics = cached_combined

    n_clusters = len(combined_topics)
    n_noise    = int((combined_labels == -1).sum())
    noise_pct  = _noise_percentage(combined_labels)
    sizes_final = _cluster_size_stats(combined_labels)
    
    print(f"[4] Final structure:")
    print(f"    → {n_clusters} clusters, {n_noise} noise ({noise_pct:.1f}%)")
    print(f"    └ Sizes: {sizes_final['min']}–{sizes_final['max']} (mean {sizes_final['mean']:.0f} ± {sizes_final['std']:.0f})")

    # ── 5. 3-D UMAP for visualisation ────────────────────────────────────────
    umap_key      = f"umap3d_{fp}"
    embeddings_3d = load_cache(umap_key)

    if embeddings_3d is None:
        print(f"[5] Computing 3-D visualization …")
        embeddings_3d = reduce_for_visualisation(embeddings, N=N)
        save_cache(umap_key, embeddings_3d)
    else:
        print(f"[5] 3-D visualization (cached)")

    # ── 6. Visualise ──────────────────────────────────────────────────────────
    print(f"[6] Building interactive visualization …")
    visualize_hierarchy(
        embeddings_3d, combined_labels, combined_topics,
        sentences, group_labels, group_topics,
    )

    # ── 7. Save cluster store for search ──────────────────────────────────────
    print(f"[7] Saving cluster store …")
    save_cluster_data(
        sentences       = sentences,
        embeddings      = embeddings,
        group_labels    = group_labels,
        group_topics    = group_topics,
        sub_labels      = sub_labels,
        combined_labels = combined_labels,
        combined_topics = combined_topics,
        out_dir         = CACHE_DIR,
    )
    
    print(f"\n{'═'*50}")
    print(f"✓ PIPELINE COMPLETE")
    print(f"{'═'*50}\n")

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Cache status
# ─────────────────────────────────────────────────────────────────────────────
import glob
existing = glob.glob(f"{CACHE_DIR}/*.pkl") + glob.glob(f"{CACHE_DIR}/*.npz") + glob.glob(f"{CACHE_DIR}/*.json")
if existing:
    total_mb = sum(os.path.getsize(p)/1e6 for p in existing)
    print(f"\nCache: {len(existing)} files ({total_mb:.1f} MB)")
else:
    print(f"\nCache: empty (will compute from scratch)")


Cache contains 7 file(s) in /content/drive/MyDrive/clustering_cache:
  cluster_data.npz                          456.0 MB
  cluster_meta.json                         17.6 MB
  combined_8f1aa04f115f.pkl                 1.0 MB
  embeddings_8f1aa04f115f.pkl               491.5 MB
  groups_8f1aa04f115f.pkl                   9.6 MB
  sub_labels_8f1aa04f115f.pkl               1.0 MB
  umap3d_8f1aa04f115f.pkl                   1.4 MB


In [9]:
main()

Loading titles …
Dataset : 120000 titles  |  fingerprint : 8f1aa04f115f
Cache   : /content/drive/MyDrive/clustering_cache

  ✓ loaded ← /content/drive/MyDrive/clustering_cache/embeddings_8f1aa04f115f.pkl
Embedding shape : (120000, 1024)
  ✓ loaded ← /content/drive/MyDrive/clustering_cache/groups_8f1aa04f115f.pkl

Level 1 — loaded 50 keyword groups
  Group  0  (68957 titles)  new, afp, talks, report, india
  Group  1  (  676 titles)  chief, chief executive, resigns, nasa chief, ex
  Group  2  (  684 titles)  oil prices, stocks, rise, barrel, crude
  Group  3  (  992 titles)  sales, retail sales, mart, wal mart, rise
  Group  4  ( 1225 titles)  win, ap, title, wrap, straight
  Group  5  (  841 titles)  killed, killed iraq, soldiers killed, gaza, baghdad
  Group  6  (  992 titles)  security, security council, microsoft, cisco, sudan
  Group  7  (  367 titles)  low, dollar, low cost, euro, hits
  Group  8  ( 1040 titles)  ibm, pc, supercomputer, new, business
  Group  9  ( 1803 titles)  ir


Saving cluster store for search …
Saving cluster arrays …
  ✓ arrays  → /content/drive/MyDrive/clustering_cache/cluster_data.npz  (456.0 MB)
Saving cluster metadata …
  ✓ metadata → /content/drive/MyDrive/clustering_cache/cluster_meta.json  (17.6 MB)

Cluster store ready:
  120,000 sentences  |  50 groups  |  453 clusters


## 📊 EVōC Cluster Visualisations
Run these cells **after `main()` completes**. They expect EVōC's outputs:
`cluster_labels_`, `cluster_layers_`, `cluster_tree_`, and the cached embeddings.

| Cell | Chart | EVōC feature used |
|------|-------|-------------------|
| A | Install + fit EVōC | `EVoC.fit_predict`, `cluster_layers_`, `cluster_tree_` |
| B | Layer Size Bar Chart | `cluster_layers_` — how granularity changes across layers |
| C | **Parallel Coordinates** | Per-cluster profile across all layers |
| D | Cluster Tree Dendrogram | `cluster_tree_` — parent/child hierarchy |
| E | 2-D UMAP Scatter (per layer) | Embedding positions coloured by each layer |
| F | Layer Transition Sankey | How clusters merge as you move up layers |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dashboard generator — loads precomputed caches and builds shareable HTML
# ─────────────────────────────────────────────────────────────────────────────
import os
import json
import numpy as np

fp = data_fingerprint(load_titles_from_csv())
meta_path = os.path.join(CACHE_DIR, "cluster_meta.json")
npz_path  = os.path.join(CACHE_DIR, "cluster_data.npz")

if not (os.path.exists(meta_path) and os.path.exists(npz_path)):
    raise FileNotFoundError(f"Cluster store not found. Run main() first.")

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

_sentences = meta.get("sentences", [])
group_topics = meta.get("group_topics", [])
combined_topics = {int(k): v for k, v in meta.get("combined_topics", {}).items()}

npz = np.load(npz_path, allow_pickle=True)
embeddings = npz["embeddings"]
group_labels = npz["group_labels"]
sub_labels = npz["sub_labels"]
combined_labels = npz["combined_labels"]

# Ensure 3-D UMAP exists
umap_key = f"umap3d_{fp}"
embeddings_3d = load_cache(umap_key)
if embeddings_3d is None:
    embeddings_3d = reduce_for_visualisation(embeddings, N=len(_sentences))
    save_cache(umap_key, embeddings_3d)

out_dir = os.path.join(CACHE_DIR, "dashboard")
os.makedirs(out_dir, exist_ok=True)

# Build interactive visualization
out_html = os.path.join(out_dir, "clusters_3d.html")
visualize_hierarchy(
    embeddings_3d, combined_labels, combined_topics,
    _sentences, group_labels, group_topics,
    output_html=out_html,
)

# Create dashboard index
index_html = os.path.join(out_dir, "index.html")
index_content = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Clustering Dashboard</title>
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <style>body{{background:#0f0f19;color:#e0e0e0;font-family:monospace}} .card{{padding:12px;margin:16px;background:#131326;border-radius:6px}}</style>
</head>
<body>
  <h2 style="padding:12px">Clustering Dashboard</h2>
  <div class="card">
    <h3>Interactive 3-D Drill-down</h3>
    <p><a href="./clusters_3d.html">Open visualiser (new tab)</a></p>
    <iframe src="./clusters_3d.html" width="100%" height="800" style="border:0"></iframe>
  </div>
</body>
</html>"""
with open(index_html, "w", encoding="utf-8") as f:
    f.write(index_content)

print(f"✓ Dashboard: {out_dir}/")


In [ ]:
# Cell replaced: visuals moved to dashboard generator cell.
# This cell previously generated an inline bar chart — now saved to dashboard/index.html
# No inline display here to keep notebook output clean.
pass


In [ ]:
# Cell replaced: parallel coordinates visualisation moved to saved dashboard.
# Inline display removed to produce shareable HTML files instead.
pass


In [ ]:
# Cell replaced: sunburst cluster tree visualisation now saved in the dashboard.
pass


In [ ]:
# Cell replaced: 2-D UMAP scatter generation moved to dashboard (no inline show).
pass


In [ ]:
# Cell replaced: Sankey visualisation moved to dashboard generator.
pass


In [10]:
# Cell replaced: semantic search / ClusterSearch initialisation left as-is, but any interactive Colab widget usage has been disabled in favor of saved dashboard files.
# To run searches use ClusterSearch pointing to CACHE_DIR (see README or cell comments).
pass


Loading cluster store from /content/drive/MyDrive/clustering_cache …
  ✓ 120,000 sentences  |  50 groups  |  453 clusters  |  embeddings (120000, 1024)

Query : 'covid vaccine side effects'
Loading encoder 'BAAI/bge-large-en-v1.5' on cuda …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

      → positive=3  negative=16  neutral=0
    [depth=2] HDBSCAN failed (Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.), returning as leaf
    [depth=2] running stance split on 19 sentences …
      → positive=2  negative=16  neutral=1
    [depth=1] HDBSCAN failed (Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.), returning as leaf
    [depth=1] running stance split on 22 sentences …
      → positive=6  negative=14  neutral=2
    [depth=1] HDBSCAN failed (Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.), returning as leaf
    [depth=1] running stance split on 30 sentences …
      → positive=6  negative=23  neutral=1
    depth=1  sim=0.592  n=   8  bird, flu, bird flu, thailand, vietnam [noise]
    depth=2  sim=0.606  n=   6  flu, bird, bird flu, pandemic, virus
    depth=2  sim=0.566  n=  19  flu, bird, bird flu, malaysia,

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


    [depth=2] pruned 18 sub-clusters (keeping top 3 of 21): sub2(sim=0.526), sub19(sim=0.498), sub4(sim=0.494), sub0(sim=0.491), sub1(sim=0.489), sub10(sim=0.486), sub15(sim=0.484), sub3(sim=0.466), sub18(sim=0.466), sub7(sim=0.462), sub12(sim=0.448), sub16(sim=0.441), sub5(sim=0.432), sub17(sim=0.424), sub8(sim=0.419), sub9(sim=0.418), sub14(sim=0.412), sub6(sim=0.403)
    [depth=3] depth limit — running stance split
    [depth=3] running stance split on 9908 sentences …
      → positive=3345  negative=5443  neutral=1120
    [depth=3] depth limit — running stance split
    [depth=3] running stance split on 451 sentences …
      → positive=113  negative=269  neutral=69
    [depth=3] depth limit — running stance split
    [depth=3] running stance split on 973 sentences …
      → positive=481  negative=371  neutral=121
    [depth=2] pruned 7 sub-clusters (keeping top 3 of 10): sub8(sim=0.494), sub6(sim=0.487), sub4(sim=0.443), sub2(sim=0.442), sub7(sim=0.435), sub0(sim=0.430), sub1(sim=0

In [11]:
# Cell replaced: interactive Colab search widget removed in notebook-run mode.
# Use the dashboard files in CACHE_DIR/dashboard for visual inspection and
# run ClusterSearch from a script or separate notebook if needed.
pass
